In [12]:
# STEP 1: Create a 3-layer MLP training pipeline.
#         Input layer:       6*L channel, position encoded coordinates [sin(2*pi*[x, y, t]), cos(2*pi*[x, y, t])], L as customized enlarging parameter 
#         Intemediate layer: 128-channel neurals
#         Output layer:      8-channel air quality data

In [13]:
import torch
import torch.nn as nn

class DanderField(nn.Module):
    # MLP
    def __init__(self, L=6):
        super().__init__()
        self.L = L
        in_dim = 3 * 2 * self.L
        self.L1 = nn.Linear(in_dim, 128)
        self.L2 = nn.Linear(128, 128)
        self.L3 = nn.Linear(128, 8)
    
    # Positional encoding
    def pos_encoding(self, coords, L=6):
        ans = []
        for l in range(L):
            ans.append(torch.sin(pow(2, l)*torch.pi*coords))
            ans.append(torch.cos(pow(2, l)*torch.pi*coords))
        return torch.cat(ans, dim=-1)

    # Pipeline
    def forward(self, coords):
        x = self.pos_encoding(coords)
        x = torch.relu(self.L1(x))
        x = torch.relu(self.L2(x))
        x = self.L3(x)
        return x

In [14]:
# Verify output shape of pipeline
model = DanderField()
out = model(torch.randn(10, 3))
print(out.shape)

torch.Size([10, 8])


In [15]:
# STEP 2a: Extract recorded data in SQLite into DataFrame
#          Failure Handling: drop data without sensor location info, print the number of row dropped
# STEP 2b: Normalization

In [16]:
import sqlite3
import pandas as pd

# Path for reveal: can be setup as contract later
DB_PATH = "../build/dander.db"
# Contract for input: 8 channels as a sequence
CHANNELS = ["pm1_0_atm", "pm2_5_atm","pm10_atm","cnt_03", "cnt_05", "cnt_10", "cnt_25", "cnt_50"]

def load_samples(db_path=DB_PATH, sensor_id=1):
    con = sqlite3.connect(db_path)
    q = """
    SELECT r.epoch_ms, l.x AS x, l.y AS y, r.pm1_0_atm, r.pm2_5_atm, r.pm10_atm, r.cnt_03, r.cnt_05, r.cnt_10, r.cnt_25, r.cnt_50
    FROM readings r
    JOIN sensor_locations l
    ON l.sensor_id = r.sensor_id
    AND r.epoch_ms >= l.valid_from
    AND (l.valid_to IS NULL OR r.epoch_ms < l.valid_to)
    WHERE r.sensor_id = ?
    ORDER BY r.epoch_ms      --reveal point: time as sequence
    """
    df = pd.read_sql(q, con, params=(sensor_id,))
    total = pd.read_sql("SELECT COUNT(*) c FROM readings WHERE sensor_id=?", con, params=(sensor_id,))["c"][0]

    con.close()
    
    # Failure Handling
    dropped = total - len(df)
    if dropped:
        print(f"WARNING: {dropped} rows are not valid and being dropped")

    return df

# Verify data in extracted dataframe
df = load_samples()
print("Sample count: ", len(df))
df.describe().loc[["min","max","mean"]].round(1).T

Sample count:  60178


,min,max,mean
epoch_ms,1.782444e+12,1.782602e+12,1.782547e+12
x,1.200000e+00,5.500000e+00,4.600000e+00
y,3.400000e+00,6.200000e+00,5.700000e+00
pm1_0_atm,0.000000e+00,2.210000e+02,1.900000e+00
pm2_5_atm,0.000000e+00,3.950000e+02,3.500000e+00
pm10_atm,0.000000e+00,5.030000e+02,4.200000e+00
cnt_03,0.000000e+00,1.782400e+04,1.970000e+02
cnt_05,0.000000e+00,1.591100e+04,1.581000e+02
cnt_10,0.000000e+00,5.088000e+03,3.720000e+01
cnt_25,0.000000e+00,6.410000e+02,3.000000e+00


In [17]:
import numpy as np
import torch

WIDTH_M, HEIGHT_M = 14.1, 7.7
t0, t1 = float(df["epoch_ms"].min()), float(df["epoch_ms"].max())

# Input Normalization
x_n = df["x"].values / WIDTH_M
y_n = df["y"].values / HEIGHT_M
t_n = (df["epoch_ms"].values - t0) / (t1 - t0)
X = np.stack([x_n, y_n, t_n], axis=1).astype("float32") # (N, 3)

# Output Normalization
Y_raw = df[CHANNELS].values.astype("float32")           # (N, 8)
Y_log = np.log1p(Y_raw)
y_mean = Y_log.mean(axis=0)
y_std = Y_log.std(axis=0) + 1e-8
Y = ((Y_log - y_mean) / y_std).astype("float32")        # (N, 8)

# Transform into torch
X = torch.from_numpy(X)
Y = torch.from_numpy(Y)
print("X: ", X.shape, "Y: ", Y.shape)
print("Y channels~0: ", Y.mean(0).round(decimals=2).tolist())
print("Y channels~1: ", Y.std(0).round(decimals=2).tolist())

# CONST for normalization, important!
norm = {"width_m": WIDTH_M, "height_m": HEIGHT_M, "t0": t0, "t1": t1,
        "y_mean": y_mean.tolist(), "y_std": y_std.tolist(), "channels": CHANNELS}

X:  torch.Size([60178, 3]) Y:  torch.Size([60178, 8])
Y channels~0:  [0.0, 0.0, -0.0, 0.0, 0.0, -0.0, 0.0, 0.0]
Y channels~1:  [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]


In [18]:
# STEP 2c: Training Loop
#          Loss:      using MSE to calculate loss in each epoch
#          Gradient:  direction and value to adjust current parameters
#          Optimizer: using Adam function to apply loss and gradient to the model

In [19]:
# Reveal model
torch.manual_seed(0)

model = DanderField()
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = torch.nn.MSELoss()

# Training loop
for epoch in range(2000):
    opt.zero_grad()          # clear last gradient
    pred = model(X)          # forward (N,3) -> (N,8)
    loss = loss_fn(pred, Y)  # calculate loss
    loss.backward()          # calculate gradient
    opt.step()               # optimizer to adjust parameters

    # Failure Handling: when learning rate is too large
    if not torch.isfinite(loss):
        print(f"Failed in Epoch {epoch}: loss is scattered.")
        break
        
    # Observation
    if epoch % 200 == 0:
        print(f"Epoch {epoch:4d} loss {loss.item():.4f}")

print("Final loss: ", loss.item())

Epoch    0 loss 1.0134
Epoch  200 loss 0.2172
Epoch  400 loss 0.1242
Epoch  600 loss 0.1048
Epoch  800 loss 0.0965
Epoch 1000 loss 0.0913
Epoch 1200 loss 0.0867
Epoch 1400 loss 0.0831
Epoch 1600 loss 0.0806
Epoch 1800 loss 0.0784
Final loss:  0.07681334018707275


In [ ]:
# STEP 3: Export .onnx
#         verify .onnx has the same output result as the pytorch model

In [ ]:
import json
import onnxscript
import onnxruntime as ort

model.eval() # close training mode and open inference mode

# export .onnx
example = torch.randn(1, 3)
N = torch.export.Dim("N")
torch.onnx.export(
    model,                                               # pytorch model
    example,                                             # sample tracing
    "../models/dander_field.onnx",                                 # output path
    input_names=["coords"],                              # name of input node, e.g. session.run(["field"], {"coords": x})
    output_names=["field"],                              # name of output node
    dynamic_shapes={"coords":{0: N}},                    # allow dynamic number in 0 dimension, s.t. we can import different number of coords
    opset_version=18,                                    # set a version to avoid change of operator behavior
)

# export norm.json
with open("../models/dander_field.norm.json", "w") as f:
    json.dump(norm, f, indent=2)

# Alignment test: ONNX == PyTorch
test = torch.randn(100, 3)
with torch.no_grad():
    torch_out = model(test).numpy()
sess = ort.InferenceSession("dander_field.onnx")
onnx_out = sess.run(["field"], {"coords": test.numpy()})[0]

max_diff = np.abs(torch_out - onnx_out).max()
print("Max difference: ", max_diff)
assert max_diff < 1e-5, "Failed: ONNX behave differently from PyTorch"
print("Success: ONNX matches PyTorch!")

/var/folders/hp/6svsxcbn39qfx9jlmfll27rr0000gn/T/ipykernel_12508/3405433069.py:9: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
  torch.onnx.export(
W0628 13:40:00.076000 12508 torch/onnx/_internal/exporter/_compat.py:133] Setting ONNX exporter to use operator set version 18 because the requested opset_version 17 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested version. If version conversion is unsuccessful, the opset version of the exported model will be kept at 18. Please consider setting opset_version >=18 to leverage latest ONNX features
W0628 13:40:00.382000 12508 torch/onnx/_internal/exporter/_registration.py:107] torchvision is not installed. Skipping torchvision::nms
W0628 13:40:00.382000 12508 torch/onnx/_int

[torch.onnx] Obtain model graph for `DanderField([...]` with `torch.export.export(..., strict=False)`...


/Users/zhu/.local/share/uv/python/cpython-3.11.15-macos-aarch64-none/lib/python3.11/copyreg.py:105: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)
The model version conversion is not supported by the onnxscript version converter and fallback is enabled. The model will be converted using the onnx C API (target version: 17).


[torch.onnx] Obtain model graph for `DanderField([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...
[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
[torch.onnx] Optimize the ONNX graph...
[torch.onnx] Optimize the ONNX graph... ✅
Max difference:  2.861023e-06
Success: ONNX matches PyTorch!
